In [1]:
from aip import AIPParameter, AIPModule, AIPLinear
import aip
import torch

In [2]:
class MLP(AIPModule):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = AIPLinear(input_dim, hidden_dim)
        self.fc2 = AIPLinear(hidden_dim, output_dim)

    def __call__(self, x):
        x = aip.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
mlp = MLP(2, 4, 1)

In [3]:
def optimizer(params, lr=0.1):
    for param in params:
        if param.grad is None:
            continue
        with torch.no_grad():
            param -= lr * param.grad
        param.grad.zero_()

In [4]:
# Test the model
def test_model(X):
    with torch.no_grad():
        y = mlp(X)
        predictions = (y > 0.5).float()
        return predictions

In [5]:
# XOR inputs and targets
X = torch.tensor(
    [[0, 0],
     [0, 1],
     [1, 0],
     [1, 1]], dtype=torch.float32
)
O = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

In [6]:
predictions = test_model(X)
print("Predictions before training:")
for i in range(len(X)):
    print(f"Input: {X[i].tolist()}, Predicted: {predictions[i].item()}, Target: {O[i].item()}")

Predictions before training:
Input: [0.0, 0.0], Predicted: 0.0, Target: 0.0
Input: [0.0, 1.0], Predicted: 0.0, Target: 1.0
Input: [1.0, 0.0], Predicted: 1.0, Target: 1.0
Input: [1.0, 1.0], Predicted: 1.0, Target: 0.0


In [7]:
# Training loop
lr = 0.1
for epoch in range(1000):
    # Forward pass
    y = mlp(X)

    # Loss: Mean Squared Error
    loss = torch.mean((y - O) ** 2)

    # Backward pass
    loss.backward()

    # Gradient descent update
    optimizer(mlp.parameters(), lr)

    # Optional: Print loss every 1000 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.4477
Epoch 100, Loss: 0.1254
Epoch 200, Loss: 0.0168
Epoch 300, Loss: 0.0004
Epoch 400, Loss: 0.0000
Epoch 500, Loss: 0.0000
Epoch 600, Loss: 0.0000
Epoch 700, Loss: 0.0000
Epoch 800, Loss: 0.0000
Epoch 900, Loss: 0.0000


In [8]:
predictions = test_model(X)
print("Predictions before training:")
for i in range(len(X)):
    print(f"Input: {X[i].tolist()}, Predicted: {predictions[i].item()}, Target: {O[i].item()}")

Predictions before training:
Input: [0.0, 0.0], Predicted: 0.0, Target: 0.0
Input: [0.0, 1.0], Predicted: 1.0, Target: 1.0
Input: [1.0, 0.0], Predicted: 1.0, Target: 1.0
Input: [1.0, 1.0], Predicted: 0.0, Target: 0.0
